# STAIR-LIA v3 — Kaggle Training Notebook
**STAIR-Enhanced v3: Local Interest Aligned (LIA)**

## Pipeline
1. **Phase 0** — Setup & Git pull
2. **Phase 1** — Copy Kaggle datasets + run `preprocess_stair_lia.py` (ZCA + ROI offline)
3. **Phase 2** — Train STAIR-LIA v3 on Baby / Sports / Electronics
4. **Phase 3** — Parse logs, display results, plot alpha evolution

> **YEU CAU**: Chay tung cell theo thu tu. Dung PAUSE neu GPU Quota het.

In [ ]:
# Cell 1: Environment Setup
import os, sys, subprocess, time, re, shutil
from pathlib import Path

KAGGLE_DATA_ROOT = Path('/kaggle/input')
WORK_DIR         = Path('/kaggle/working')
REPO_DIR         = WORK_DIR / 'STAIR-Enhanced'
LOG_DIR          = WORK_DIR / 'logs_lia_v3'
PREPROC_DIR      = WORK_DIR / 'preprocessed_lia'
LOG_DIR.mkdir(exist_ok=True)
PREPROC_DIR.mkdir(exist_ok=True)

# --- Dataset slugs ---
DATASETS = [
    {
        'key':  'baby',
        'name': 'Amazon2014Baby_550_MMRec',
        'slug': 'amazon2014baby550mmrec',
        'cfg':  'Amazon2014Baby_550_MMRec.yaml',
        'bs':   '1024',
        'wd':   '0.3',
        'gamma':'0.1',
    },
    {
        'key':  'sports',
        'name': 'Amazon2014Sports_550_MMRec',
        'slug': 'amazon2014sports550mmrec',
        'cfg':  'Amazon2014Sports_550_MMRec.yaml',
        'bs':   '1024',
        'wd':   '0.1',
        'gamma':'0.2',
    },
    {
        'key':  'electronics',
        'name': 'Amazon2014Electronics_550_MMRec',
        'slug': 'amazon2014electronics550mmrec',
        'cfg':  'Amazon2014Electronics_550_MMRec.yaml',
        'bs':   '4096',
        'wd':   '0.1',
        'gamma':'0.4',
    },
]

print('[Setup] Kaggle STAIR-LIA v3 Training Environment')
print(f'  WORK_DIR    : {WORK_DIR}')
print(f'  REPO_DIR    : {REPO_DIR}')
print(f'  LOG_DIR     : {LOG_DIR}')
print(f'  PREPROC_DIR : {PREPROC_DIR}')
import torch
print(f'  CUDA        : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU         : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM        : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


In [ ]:
# Cell 2: Clone / Update STAIR-Enhanced repository
if REPO_DIR.exists():
    print('Updating existing repository...')
    result = subprocess.run(
        ['git', 'pull'],
        capture_output=True, text=True, cwd=str(REPO_DIR)
    )
    print(result.stdout)
    if result.returncode != 0:
        print('Pull failed, re-cloning...')
        shutil.rmtree(str(REPO_DIR))
        subprocess.run(
            ['git', 'clone', 'https://github.com/ThanhChuong12/STAIR-Enhanced.git', str(REPO_DIR)],
            check=True
        )
else:
    print('Cloning repository...')
    subprocess.run(
        ['git', 'clone', 'https://github.com/ThanhChuong12/STAIR-Enhanced.git', str(REPO_DIR)],
        check=True
    )

print('\nVerify key files:')
for f in ['main_stair_lia_v3.py', 'preprocess_stair_lia.py',
          'models/residual_projector_v2.py']:
    exists = (REPO_DIR / f).exists()
    print(f'  {f}: {"OK" if exists else "MISSING"}')


In [ ]:
# Cell 3: Copy Kaggle Dataset Files -> /kaggle/data/
DATA_ROOT = Path('/kaggle/data')
DATA_ROOT.mkdir(exist_ok=True)
(DATA_ROOT / 'Processed').mkdir(exist_ok=True)

for ds in DATASETS:
    src = KAGGLE_DATA_ROOT / ds['slug'] / ds['name']
    dst = DATA_ROOT / 'Processed' / ds['name']
    if not src.exists():
        print(f'  WARNING NOT FOUND: {src}  (check Dataset input slug)')
        continue
    dst.mkdir(parents=True, exist_ok=True)
    for f in src.iterdir():
        shutil.copy2(str(f), str(dst / f.name))
    print(f'  [OK] {ds["name"]}: {len(list(dst.iterdir()))} files')

# Update configs root
for ds in DATASETS:
    cfg_path = REPO_DIR / 'configs' / ds['cfg']
    if cfg_path.exists():
        txt = cfg_path.read_text(encoding='utf-8')
        txt = txt.replace('root: data', 'root: /kaggle/data/Processed')
        txt = txt.replace('root: ../../data', 'root: /kaggle/data/Processed')
        cfg_path.write_text(txt, encoding='utf-8')
        print(f'  [Config] root updated: {cfg_path.name}')

print('[OK] Dataset files ready.')


In [ ]:
# Cell 4: Phase 1 -- Offline ZCA Whitening + Bidirectional ROI Extraction
# Chay mot lan de tao E_fused_t.pt / E_fused_v.pt / E_roi_t.pt / E_roi_v.pt
# TOAN BO OFFLINE -- Khong train, khong gradient

PREPROC_SCRIPT = str(REPO_DIR / 'preprocess_stair_lia.py')

def run_preprocessing(ds):
    ds_name = ds['name']
    ds_dir  = DATA_ROOT / 'Processed' / ds_name
    out_dir = PREPROC_DIR

    # Tim file features
    text_path   = None
    visual_path = None
    for fname in ds_dir.iterdir():
        n = fname.name.lower()
        if 'text' in n and fname.suffix in ('.pt', '.npy', '.pkl'):
            text_path = str(fname)
        if ('vis' in n or 'image' in n or 'img' in n) and fname.suffix in ('.pt', '.npy', '.pkl'):
            visual_path = str(fname)

    if text_path is None or visual_path is None:
        print(f'  [{ds["key"]}] WARNING: textual/visual feat not found, skipping')
        print(f'  Files in dir: {[f.name for f in ds_dir.iterdir()]}')
        return

    print(f'\n--- Preprocessing: {ds["key"]} ---')
    print(f'  text:   {text_path}')
    print(f'  visual: {visual_path}')

    cmd = [
        sys.executable, PREPROC_SCRIPT,
        '--dataset',      ds_name,
        '--text_feat',    text_path,
        '--visual_feat',  visual_path,
        '--output_dir',   str(out_dir),
        '--d_sub',        '64',
        '--d_k',          '32',
        '--fuweight',     '0.6',
        '--batch_size',   '512',
        '--device',       'cuda' if torch.cuda.is_available() else 'cpu',
        '--sanity_check',
    ]
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - t0
    if result.returncode == 0:
        print(f'  [OK] Preprocessing done in {elapsed:.0f}s')
        out_ds = out_dir / ds_name
        for fname in sorted(out_ds.iterdir()):
            size_mb = fname.stat().st_size / 1e6
            print(f'    {fname.name}: {size_mb:.1f} MB')
    else:
        print(f'  [FAIL] Exit {result.returncode}')
        print(result.stdout[-2000:])
        print(result.stderr[-1000:])

# Chay preprocessing cho ca 3 dataset
for ds in DATASETS:
    run_preprocessing(ds)

print('\n[Cell 4 DONE] Preprocessing complete.')


In [ ]:
# Cell 5: Copy preprocessed_lia/ -> /kaggle/data/Processed/{ds}/preprocessed_lia/
# (De main_stair_lia_v3.py co the load duoc qua cfg.lia_precomputed_dir)

for ds in DATASETS:
    src_dir = PREPROC_DIR / ds['name']
    dst_dir = DATA_ROOT / 'Processed' / ds['name'] / 'preprocessed_lia' / ds['name']
    if not src_dir.exists():
        print(f'  [{ds["key"]}] No precomputed dir found, will use SVD fallback')
        continue
    dst_dir.mkdir(parents=True, exist_ok=True)
    for f in src_dir.iterdir():
        shutil.copy2(str(f), str(dst_dir / f.name))
    print(f'  [OK] {ds["key"]}: {len(list(dst_dir.iterdir()))} files -> {dst_dir}')

print('[Cell 5 DONE] E_fused files staged for training.')


In [ ]:
# Cell 6: Training Helper Functions

import re, json

def parse_best_result(log_path):
    """Trich xuat ket qua tot nhat tu log file (Recall@20, NDCG@20)."""
    if not Path(log_path).exists():
        return None, None
    content = Path(log_path).read_text(encoding='utf-8', errors='ignore')
    pattern = r'Recall@10=([0-9.]+).*?Recall@20=([0-9.]+).*?NDCG@10=([0-9.]+).*?NDCG@20=([0-9.]+)'
    matches = re.findall(pattern, content)
    if not matches:
        return None, None
    best_ndcg20 = max(float(m[3]) for m in matches)
    best_m = max(matches, key=lambda m: float(m[3]))
    best_ep_m = re.findall(r'Epoch\s*(\d+)', content)
    return {
        'Recall@10': float(best_m[0]),
        'Recall@20': float(best_m[1]),
        'NDCG@10':   float(best_m[2]),
        'NDCG@20':   float(best_m[3]),
    }, None

def parse_alpha_history(log_path):
    """Trich xuat lich su alpha (text weight) qua cac epoch."""
    if not Path(log_path).exists():
        return []
    content = Path(log_path).read_text(encoding='utf-8', errors='ignore')
    matches = re.findall(r'alpha @epoch\s*(\d+).*?([0-9.]+)', content)
    return [(int(ep), float(val)) for ep, val in matches]

def run_training(ds, extra_args=None):
    """Chay training cho 1 dataset. Khong block -- log ra file."""
    key      = ds['key']
    cfg_file = str(REPO_DIR / 'configs' / ds['cfg'])
    log_file = str(LOG_DIR / f'{key}.log')

    print(f'\n{"="*60}')
    print(f'STAIR-LIA v3 Training: {key.upper()}')
    print(f'Config : {cfg_file}')
    print(f'Log    : {log_file}')
    print(f'{"="*60}')

    cmd = [
        sys.executable,
        str(REPO_DIR / 'main_stair_lia_v3.py'),
        '--config', cfg_file,
        '--lia-precomputed-dir', 'preprocessed_lia',
        '--lia-alpha',     '0.5',
        '--lia-roi-cl',    '0.01',   # Enable nhẹ ROI contrastive
        '--lia-temperature', '0.07',
    ]
    if extra_args:
        cmd.extend(extra_args)

    t0 = time.time()
    with open(log_file, 'w', encoding='utf-8') as lf:
        proc = subprocess.run(
            cmd,
            stdout=lf, stderr=subprocess.STDOUT,
            cwd=str(REPO_DIR)
        )
    elapsed = (time.time() - t0) / 60.0

    if proc.returncode == 0:
        print(f'[OK] {key.upper()} done in {elapsed:.1f} min')
        best, _ = parse_best_result(log_file)
        if best:
            print(f'  Best Recall@20={best["Recall@20"]:.4f} | NDCG@20={best["NDCG@20"]:.4f}')
    else:
        print(f'[FAIL] {key.upper()} exit={proc.returncode} ({elapsed:.1f} min)')
        log_tail = Path(log_file).read_text(encoding='utf-8', errors='ignore').splitlines()
        print('Last 20 lines:')
        print('\n'.join(log_tail[-20:]))
    return proc.returncode

print('[Cell 6] Helper functions defined.')


In [ ]:
# Cell 7: Train STAIR-LIA v3 -- BABY
# Estimated time: ~30-45 min on T4
ret = run_training(DATASETS[0])
print(f'Return code: {ret}')


In [ ]:
# Cell 8: Train STAIR-LIA v3 -- SPORTS
# Estimated time: ~60-80 min on T4
ret = run_training(DATASETS[1])
print(f'Return code: {ret}')


In [ ]:
# Cell 9: Train STAIR-LIA v3 -- ELECTRONICS
# Estimated time: ~90-120 min on T4
ret = run_training(DATASETS[2])
print(f'Return code: {ret}')


In [ ]:
# Cell 10: Results Summary -- STAIR-LIA v3
BASELINE = {
    'baby':        {'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
    'sports':      {'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
    'electronics': {'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
}

print(f'{'='*80}')
print(f'{'Dataset':15} {'Metric':12} {'Baseline':>10} {'LIA_v3':>10} {'Delta':>10}')
print(f'{'='*80}')

all_results = {}
for ds in DATASETS:
    key = ds['key']
    log_file = str(LOG_DIR / f'{key}.log')
    best, _ = parse_best_result(log_file)
    all_results[key] = best
    if best is None:
        print(f'{key:15} NO RESULT')
        continue
    for metric in ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']:
        bl   = BASELINE[key][metric]
        v3   = best[metric]
        delt = (v3 - bl) / bl * 100
        flag = '[+]' if delt > 0 else '[-]'
        print(f'{key:15} {metric:12} {bl:>10.4f} {v3:>10.4f} {delt:>+9.2f}% {flag}')
    print()

print(f'{'='*80}')


In [ ]:
# Cell 11: Plot Alpha Evolution (Text vs Visual weight)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('STAIR-LIA v3: Alpha (Text Weight) Evolution', fontsize=14, fontweight='bold')

for i, ds in enumerate(DATASETS):
    key = ds['key']
    log_file = str(LOG_DIR / f'{key}.log')
    history = parse_alpha_history(log_file)
    ax = axes[i]
    if history:
        epochs  = [h[0] for h in history]
        alphas  = [h[1] for h in history]
        ax.plot(epochs, alphas, 'b-', linewidth=2, label='alpha (text)')
        ax.plot(epochs, [1 - a for a in alphas], 'r--', linewidth=1.5, label='1-alpha (visual)')
        ax.axhline(y=0.5, color='gray', linestyle=':', label='init=0.5')
        ax.set_ylim(0, 1)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Weight')
        ax.set_title(f'{key.capitalize()}')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    else:
        ax.set_title(f'{key.capitalize()} (No data)')

plt.tight_layout()
out_fig = str(WORK_DIR / 'lia_v3_alpha_evolution.png')
plt.savefig(out_fig, dpi=150)
plt.show()
print(f'[Saved] {out_fig}')


In [ ]:
# Cell 12: Plot Training Loss Curves
import matplotlib.pyplot as plt, re

def parse_loss_curve(log_path):
    if not Path(log_path).exists():
        return [], []
    content = Path(log_path).read_text(encoding='utf-8', errors='ignore')
    matches = re.findall(r'Epoch\s*(\d+).*?LOSS=([0-9.]+)', content)
    epochs  = [int(m[0]) for m in matches]
    losses  = [float(m[1]) for m in matches]
    return epochs, losses

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('STAIR-LIA v3: Training Loss Curves', fontsize=14, fontweight='bold')

for i, ds in enumerate(DATASETS):
    key = ds['key']
    log_file = str(LOG_DIR / f'{key}.log')
    eps, losses = parse_loss_curve(log_file)
    ax = axes[i]
    if eps:
        ax.plot(eps, losses, 'g-', linewidth=1.5)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('BPR Loss')
        ax.set_title(f'{key.capitalize()}')
        ax.grid(True, alpha=0.3)
    else:
        ax.set_title(f'{key.capitalize()} (No data)')

plt.tight_layout()
out_fig = str(WORK_DIR / 'lia_v3_loss_curves.png')
plt.savefig(out_fig, dpi=150)
plt.show()
print(f'[Saved] {out_fig}')
